In [ ]:
# ==============================================================================
# 3-TIMELINE BV / AV / LV  --  UW / ST / WS prevalence at three timepoints
# ==============================================================================
# Runs as Stage 5 of the crosstabs pipeline. master_pipeline.py rewrites the two
# constants below to this run's paths; the values here are only what the
# notebook opens with when an analyst runs it by hand.
#
# INPUT_FILE is the Stage 2 derived workbook (XX_DDMMYY_Exc_V(DDMMYY)_2.xlsx).
# OUTPUT_DIR is a folder, not a file -- the workbook name is derived from the
# input's own dataset code and version stamp further down, so an output is
# always named after the input it came from.
import os
import re

import pandas as pd

INPUT_FILE = "UJ_020926_Exc_V(020926)_2.xlsx"
OUTPUT_DIR = "outputs/bv_av_lv"


In [ ]:
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {os.path.basename(str(INPUT_FILE))}: {df.shape[0]} rows x {df.shape[1]} columns")
df.shape


In [ ]:
# Analysis population. Both gates must be present -- silently skipping a missing
# one would change the denominator of every table below without saying so.
for _gate in ("Primary Exclusion PNC 2", "pre_analysis_exclusion"):
    if _gate not in df.columns:
        raise KeyError(
            f"Column '{_gate}' is not in {os.path.basename(str(INPUT_FILE))}. "
            "The BV/AV/LV tables are defined on the twice-filtered population, "
            "so this cannot be skipped -- check that Stage 2 produced it."
        )

df = df[df["Primary Exclusion PNC 2"] == "Included"]
df = df[df["pre_analysis_exclusion"] == "Included"]
print(f"After exclusions: {df.shape[0]} rows")
df.shape


In [ ]:
# ---- Dynamic output naming (derived from INPUT_FILE; naming only) ------------
# Same convention as the other crosstab stages: carry the input's dataset code
# (XX_DDMMYY) and version stamp (V(DDMMYY)) into the output name, so a workbook
# can always be traced back to the extract it came from. Falls back to a plain
# name if the input is not named to the convention.
_bvl_base = os.path.basename(str(INPUT_FILE))
_bvl_code = re.search(r"([A-Za-z]{2}_\d{6})", _bvl_base)
_bvl_ver = re.search(r"V\((\d{6})\)", _bvl_base)

_bvl_stem = f"{_bvl_code.group(1).upper()}_BV_AV_LV" if _bvl_code else "BV_AV_LV"
if _bvl_ver:
    _bvl_stem += f"_V({_bvl_ver.group(1)})"

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{_bvl_stem}.xlsx")
print(f"  Output workbook: {OUTPUT_FILE}")
TIMEPOINTS = ["BV", "AV", "LV"]

INDICATOR_COLS = {
    "UW": {tp: f"WFA status at {tp} 2" for tp in TIMEPOINTS},
    "ST": {tp: f"HFA status at {tp} 2" for tp in TIMEPOINTS},
    "WS": {tp: f"WFH status at {tp} 2" for tp in TIMEPOINTS},
}
COMBO_CATEGORIES = {"UW": ["SUW", "MUW"], "ST": ["SST", "MST"], "WS": ["MAM", "SAM"]}
DISPLAY_MODES = [("n_pct", "n (%)"), ("pct_only", "%"), ("n_only", "n")]

TYPE_OF_ADOPTION_COL = "Type of adoption"

BREAKDOWN_VARS = [
    "Blocks", "Role Group", "Department", "Learner Category 2",
    "Total Adoptions group", "Total Adoptions group after exclusion",
    "Primary Exclusion ANC", "Primary Exclusion PNC 2", "Type of adoption",
    "Adoption type 1", "Mother Age Group",
    "Is this the mother\'s first pregnancy_C", "Number of child\'s siblings_C",
    "Mother\'s education level_M", "Color of mother\'s family ration card_M",
    "Mother\'s family type_M", "Mother\'s social category_M",
    "Location of Delivery category", "Method of delivery_C",
    "Mother adoption till birth", "gestational week classification",
    "adoption age classification", "last visit age classification 1",
    "last visit age classification 2", "adoption duration baby",
    "visit category", "bf assessment category", "cf assessment category",
    "Activity Score", "birthweight category", "birthweight category 2",
    "WFA status at BV 2", "WFA status at AV 2", "WFA status at LV 2",
    "WFA at LV/Change zscore LV AV", "HFA status at BV 2", "HFA status at AV 2",
    "HFA status at LV 2", "WFH status at BV 2", "WFH status at AV 2",
    "WFH status at LV 2", "Member Tag CR", "Followup Category",
    "avg weight gain 28g 60pct YN", "avg weight gain 17g 60pct YN",
    "Catchup YN", "Avg Weight Gain Category LV AV 2", "Avg Weight Gain Category LV AV",
    "Learner Normal 60pct", "Improvement in WFA Zscore btw AV LV",
    "ANC 60d 3visit", "PNC lt5 60d 8visit", "PNC gt5 60d 4visit",
]

In [ ]:
row_order = {
    "Department": ['HFW', 'WCD', 'MSRLS'],
    "District": ['East Garo Hills', 'East Jaintia Hills', 'East Khasi Hills', 'Eastern West Khasi Hills',
                 'North Garo Hills', 'Ri Bhoi', 'South Garo Hills', 'South West Garo Hills',
                 'South West Khasi Hills', 'West Garo Hills', 'West Jaintia Hills', 'West Khasi Hills'],
    "Role Group": ['AWW', 'ASHA', 'ASHASup', 'ANM', 'CHO', 'Other'],
    "Member Tag_CR": ['Master Trainer', 'Facilitator', 'Other', 'Under evaluation', 'Under Evaluation'],
    "Total Adoptions group": ['01_to_03', '04_to_06', '07_to_09', '10_or_more'],
    "Total Adoptions group after exclusion": ['01_to_03', '04_to_06', '07_to_09', '10_or_more'],
    "Type of adoption": ['ANC', 'PNCL5M', 'PNCG5M', 'Data to calculate child age of adoption is not available',
                          'Invalid difference: Baby Adoption date earlier than Mother adoption date',
                          'Mother adoption date is missing'],
    "Mother Age Group": ['15_to_18_years_old', '19_to_24_years_old', '25_to_36_years_old',
                          '37_to_46_years_old', '47_to_50_years_old'],
    "Is this the mother's first pregnancy_C": ["Yes", "No"],
    "Mother's education level_M": ['Illiterate', 'Class 5', 'Class 8', 'Class 10', 'Class 12',
                                    'Vocational education', 'Graduate', 'Post-graduate'],
    "Color of mother's family ration card_M": ['Yellow', 'Orange', 'White', 'Pink', 'No ration card', 'Other'],
    "Mother's family type_M": ['Joint family', 'Nuclear family'],
    "Mother's social category_M": ['General', 'Other Backward Class (OBC)', 'Scheduled Caste (SC)',
                                    'Scheduled Tribe (ST)'],
    "Location of Delivery category": ['Government', 'Private', 'Home'],
    "Method of delivery_C": ['Normal', 'Cesarean Section', 'Assisted delivery'],
    "birthweight category": ['01.5_or_less_kg', '01.51_to_02.5_kg', '02.51_to_02.7_kg', '02.71_to_03.0_kg',
                              '03.01_to_03.5_kg', 'More_than_03.5_kg'],
    "birthweight category 2": ['Less_than_01.5_kg', '01.5_to_02.49_kg', '02.5_to_02.69_kg', '02.7_to_02.99_kg',
                                '03.0_to_03.49_kg', '03.5_or_more_kg', 'Above_+6SD'],
    "birthweight group": ['Low Birth Weight', 'Normal Birth Weight', 'Overweight', 'Birth Weight is not available'],
    "adoption age classification": ['invalid_age', 'd000', 'd001_to_d015', 'd016_to_d030', 'd031_to_d060',
                                     'd061_to_d090', 'd091_to_d120', 'd121_to_d150', 'd151_to_d180',
                                     'd181_to_d210', 'd211_to_d240', 'd241_to_d270', 'd271_to_d300',
                                     'd301_to_d330', 'd331_to_d365', 'd366_plus'],
    "last visit age classification 1": ['d001_to_d015', 'd016_to_d030', 'd031_to_d060', 'd061_to_d090',
                                         'd091_to_d120', 'd121_to_d150', 'd151_to_d180', 'd181_to_d210',
                                         'd211_to_d240', 'd241_to_d270', 'd271_to_d300', 'd301_to_d330',
                                         'd331_to_d365', 'd366_plus'],
    "last visit age classification 2": ['d001_to_d060', 'd061_to_d120', 'd121_to_d180', 'd181_to_d210',
                                         'd211_to_d240', 'd241_to_d270', 'd271_to_d300', 'd301_to_d330',
                                         'd331_to_d360', 'd361_plus'],
    "adoption duration baby": ['Only birth data available', '01_to_30_days', '31_to_60_days', '61_to_90_days',
                                '91_days_or_more', 'negatives'],
    "visit category": ['01_visit', '02_visits', '03_visits', '04_to_05_visits', '06_to_07_visits',
                        '08_to_09_visits', '10_or_more_visits'],
    "bf assessment category": ['00_assessment', '01_assessment', '02_assessments', '03_assessments',
                                '04_to_05_assessments', '06_to_07_assessments', '08_to_09_assessments',
                                '10_or_more_assessments'],
    "cf assessment category": ['00_assessment', '01_assessment', '02_assessments', '03_assessments',
                                '04_to_05_assessments', '06_to_07_assessments', '08_to_09_assessments',
                                '10_or_more_assessments'],
    "bf assessment category 2": ['00_assessment', '01_assessment', '02_assessments', '03_assessments',
                                  '04_to_05_assessments', '06_or_more_assessments'],
    "cf assessment category 2": ['00_assessment', '01_assessment', '02_assessments', '03_assessments',
                                  '04_to_05_assessments', '06_or_more_assessments'],
    "Activity Score": ['00', '01_to_02', '03_to_10', '11_to_20', '21_to_30', '31_or_above'],
    "WFA status at BV 2": ['Normal', 'MUW', 'SUW', 'Zscore not available'],
    "WFA status at AV 2": ['Normal', 'MUW', 'SUW', 'Zscore not available'],
    "WFA status at LV 2": ['Normal', 'MUW', 'SUW', 'Zscore not available'],
    "HFA status at BV 2": ['Normal', 'MST', 'SST', 'Zscore not available'],
    "HFA status at AV 2": ['Normal', 'MST', 'SST', 'Zscore not available'],
    "HFA status at LV 2": ['Normal', 'MST', 'SST', 'Zscore not available'],
    "WFH status at BV 2": ['Normal', 'MAM', 'SAM', 'Zscore not available'],
    "WFH status at AV 2": ['Normal', 'MAM', 'SAM', 'Zscore not available'],
    "WFH status at LV 2": ['Normal', 'MAM', 'SAM', 'Zscore not available'],
    "WFA at LV/Change zscore LV AV": ['Normal/Catchup', 'Normal/nan', 'None/nan', 'Normal/0.00 to 0.67',
                                       'MUW/0.00 to 0.67', 'MUW/Faltering', 'Mild/0.00 to 0.67',
                                       'Mild/-0.67 to -0.01', 'Mild/Catchup', 'Normal/-0.67 to -0.01',
                                       'Mild/Faltering', 'Normal/Faltering', 'SUW/Faltering',
                                       'MUW/-0.67 to -0.01', 'MUW/Catchup', 'Mild/nan',
                                       'SUW/0.00 to 0.67', 'SUW/Catchup', 'MUW/nan', 'SUW/-0.67 to -0.01',
                                       'SUW/nan'],
    "number of protein assessment category": ['00_assessment', '01_assessment', '02_assessments',
                                                '03_assessments', '04_to_05_assessments', '06_to_07_assessments',
                                                '08_to_09_assessments', '10_or_more_assessments'],
    "gestational week classification": ['20 weeks or less', '20.1 to 28 weeks', '28.1 to 34 weeks',
                                         '34.1 to 36.9 weeks', '37 to 41.9 weeks', '42 weeks or more'],
    "Mother adoption till birth": ["PNC", "001_to_030", "031_to_060", "061_to_090", "091_to_120",
                                    "121_to_150", "151_to_180", "181_to_210", "211_to_240", "241_to_270",
                                    "271_to_300", 'DOB is missing', 'Mother adoption date is missing',
                                    'DOB and Mother adoption both missing',
                                    'Mother adoption during pregnancy is More than 300 days- Suggestive of data entry error'],
    "Adoption type 1": ['ANC>=60D', 'ANC<60D', 'PNCL5M', 'PNCG5M'],
    'ANC 60d 3visit': ['Yes', 'No'],
    'PNC lt5 60d 8visit': ['Yes', 'No'],
    'PNC gt5 60d 4visit': ['Yes', 'No'],
    'Included excluded': ['Included', 'Excluded'],
    'Primary Exclusion PNC 2': [
        "No Child ID - Child not born yet/child not followed up", "Birth date not available",
        "Adoption visit anthropometry data(Wt,Ht,zscore) not available",
        "Mother adoption date is after last visit date of the child - suggestive of data entry error",
        "Birth weight category is below -6SD or above +6SD",
        "Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match",
        "Date of birth in CR sheets and Visit Date 1 in CM sheet do not match",
        "Adoption date in CR sheet and Visit Date 2 in CM sheet do not match",
        "Last visit dates do not match in CR and CM sheets",
        "Wt Gain per day \u2265 180gm between Adoption and Last visit", "Only adoption visit occurred",
        "Only birth anthropometry details available", "mother_AD_DOB > 300",
        "Birth date, Adoption date, and Last visit date are the same",
        'User role belongs to excluded category', 'User Role is not available',
        "Weight Zscore at LV/AV/BV is missing", "Height Zscore at LV/AV/BV is missing",
        "WFH Zscore at LV/AV/BV is missing",
        'Invalid difference: Baby Adoption date earlier than Mother adoption date',
        "Follow-up category is Unclassified ", "Last visit Date not available", 'Included'],
    'Primary Exclusion ANC': [
        "No Child ID - Child not born yet/child not followed up", "Birth date not available",
        "Birth weight category is below -6SD or above +6SD",
        "Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match",
        "Date of birth in CR sheets and Visit Date 1 in CM sheet do not match",
        'User role belongs to excluded category', 'User Role is not available',
        "Weight Zscore at LV/AV/BV is missing", "Height Zscore at LV/AV/BV is missing",
        "WFH Zscore at LV/AV/BV is missing", 'Follow-up category is Unclassified',
        'Invalid difference: Baby Adoption date earlier than Mother adoption date',
        "Follow-up category is Unclassified ", "Last visit Date not available", 'Included'],
    "Avg Weight Gain Category LV AV 2": ['0 or less', '0.1\u20135 g/d', '5.1\u201310 g/d', '10.1\u201315 g/d',
                                          '15.1\u201317 g/d', '17.1\u201320 g/d', '20.1\u201325 g/d',
                                          '25.1\u201328 g/d', '28.1\u201330 g/d', '30\u201335 g/d',
                                          '35.1\u201340 g/d', 'More than 40 g/d'],
    "Avg Weight Gain Category LV AV": ['Less than 0g/d', '0.1_to_10g/d', '10.1_to_20g/d', '20.1_to_30g/d',
                                        '30.1_to_40g/d', '40.1_to_50g/d', 'More than 50g/d'],
    "Learner Normal 60pct": ["Yes", "No"],
    "Improvement in WFA Zscore btw AV LV": ["Yes", "No"],
    "Catchup YN": ["Yes", "No"],
    "avg weight gain 17g 60pct YN": ['Yes', 'No'],
    "avg weight gain 28g 60pct YN": ['Yes', 'No'],
}
row_order["Blocks"] = ['Ambad', 'Badnapur', 'Bhokardan', 'Ghansawangi', 'Jafrabad', 'Jalna', 'Mantha', 'Partur']
row_order["Total Adoptions group jalna"] = ['01', '02', '03', 'More_than_03']
row_order["Total Adoptions group after exclusion jalna"] = ['01', '02', '03', 'More_than_03']
row_order["Learner Category 3"] = ['MT + FL', 'Learner', 'Other', 'Under Evaluation', 'Under evaluation']


def order_categories(var, cats):
    """Order category values for a breakdown variable using row_order,
    falling back to alphabetical order for variables/values not listed."""
    cats = [str(c) for c in cats]
    if var in row_order:
        fixed_order = row_order[var]
        in_order = [c for c in fixed_order if c in cats]
        leftover = sorted(set(cats) - set(fixed_order))
        return in_order + leftover
    return sorted(cats)

In [ ]:
def fmt(cnt, n_total, mode):
    pct = round(100 * cnt / n_total, 1) if n_total else 0.0
    if mode == "n_pct":
        return f"{cnt} ({pct}%)"
    elif mode == "pct_only":
        return f"{pct}%"
    return f"{cnt}"


def build_count_rows(data, status_cols, category_values, breakdown_vars):
    """Compute raw counts ONCE per (sheet, indicator) using a single
    groupby().agg() pass per breakdown variable -- not one boolean filter
    of the full wide dataframe per category value."""
    status_col_list = list(status_cols.values())
    needed = list(dict.fromkeys(status_col_list + [v for v in breakdown_vars if v in data.columns]))
    slim = data[needed].copy()

    hit = slim[status_col_list].isin(category_values)
    hit.columns = [f"__hit_{c}" for c in status_col_list]
    combined = pd.concat([slim, hit], axis=1)

    raw_rows = [("ROW", "All", len(slim),
                 {tp: int(hit[f"__hit_{status_cols[tp]}"].sum()) for tp in TIMEPOINTS}),
                ("BLANK",)]

    for var in breakdown_vars:
        if var not in slim.columns:
            raw_rows.append(("MISSING", var))
            raw_rows.append(("BLANK",))
            continue

        raw_rows.append(("HEADER", var))
        sub = combined.dropna(subset=[var])
        if len(sub):
            agg = sub.groupby(var, observed=True).agg(
                n=(var, "size"),
                **{tp: (f"__hit_{status_cols[tp]}", "sum") for tp in TIMEPOINTS},
            )
            agg.index = agg.index.astype(str)
            cats = order_categories(var, agg.index.tolist())
            for c in cats:
                if c not in agg.index:
                    continue
                r = agg.loc[c]
                raw_rows.append(("ROW", f"    {c}", int(r["n"]),
                                  {tp: int(r[tp]) for tp in TIMEPOINTS}))
        raw_rows.append(("BLANK",))
    return raw_rows


def rows_to_table(raw_rows, mode):
    out = []
    for r in raw_rows:
        if r[0] == "BLANK":
            out.append({"Category": "", "n": "", "BV": "", "AV": "", "LV": ""})
        elif r[0] == "HEADER":
            out.append({"Category": r[1], "n": "", "BV": "", "AV": "", "LV": ""})
        elif r[0] == "MISSING":
            out.append({"Category": f"[MISSING COLUMN: {r[1]}]", "n": "", "BV": "", "AV": "", "LV": ""})
        else:
            _, label, n_total, cnts = r
            row = {"Category": label, "n": n_total}
            for tp in TIMEPOINTS:
                row[tp] = fmt(cnts[tp], n_total, mode)
            out.append(row)
    return pd.DataFrame(out)


def make_sheets(df):
    sheets = {"All": df}
    if TYPE_OF_ADOPTION_COL in df.columns:
        sheets["ANC"] = df[df[TYPE_OF_ADOPTION_COL] == "ANC"]
        sheets["PNCL5M"] = df[df[TYPE_OF_ADOPTION_COL] == "PNCL5M"]
        sheets["PNCG5M"] = df[df[TYPE_OF_ADOPTION_COL] == "PNCG5M"]
        sheets["ANC_or_PNCL5M"] = df[df[TYPE_OF_ADOPTION_COL].isin(["ANC", "PNCL5M"])]
    else:
        print(f"WARNING: column '{TYPE_OF_ADOPTION_COL}' not found - only 'All' sheet will be built.")
    return sheets


def export(df, output_file=OUTPUT_FILE):
    sheets = make_sheets(df)
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet_name, data_subset in sheets.items():
            row_cursor = 1
            for indicator, status_cols in INDICATOR_COLS.items():
                combo = COMBO_CATEGORIES[indicator]
                raw_rows = build_count_rows(data_subset, status_cols, combo, BREAKDOWN_VARS)

                col_cursor = 0
                max_rows = 0
                for mode, mode_label in DISPLAY_MODES:
                    table = rows_to_table(raw_rows, mode)
                    table.to_excel(writer, sheet_name=sheet_name, startcol=col_cursor,
                                    startrow=row_cursor, index=False)
                    ws = writer.sheets[sheet_name]
                    ws.cell(row=row_cursor, column=col_cursor + 1,
                             value=f"{indicator} - {'+'.join(combo)} - {mode_label}")
                    col_cursor += table.shape[1] + 1
                    max_rows = max(max_rows, table.shape[0])
                row_cursor += max_rows + 3
    print(f"Done. Saved to {output_file}")

In [ ]:
export(df)